# **AI Agents in LangChain**

In [164]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
import requests

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "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"

## **1. Simple AI Agent**

In [168]:
# Wrap tool properly
search = DuckDuckGoSearchRun()

# Step 1: Define tool
@tool
def search_tool(query: str) -> str:
    """Search the web for current information using DuckDuckGo."""

    result = search.run(query)

    return f"""
    Extract the final answer from this:

    {result}

    Give a clear final answer.
    """

# Step 2: Create LLM (Groq)
llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

# Step 3: Create Agent
agent = create_agent(
    model=llm,
    tools=[search_tool],
    system_prompt="""
    You are a reliable AI assistant.

    Rules:
    - Use tools when needed
    - AFTER getting tool result → STOP searching
    - TRUST the tool result
    - Give final answer clearly
    - DO NOT search multiple times for same question
    - Print the Final Answer as Output only.
"""
)

# Step 4: Invoke
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "Top News in Pakistan Today"}
    ]
})

In [169]:
# Step 5: Print the Output
print(response["messages"][-1].content)

The top news in Pakistan today includes:

- A rainbow appears in Karachi after heavy rains.
- A death toll of six people is reported due to the rains.
- A new western weather system is expected.
- An air strike in Kabul kills at least 100 people at a drug rehab centre.
- Pakistanis are feeling the effects of fuel price hikes due to the Middle East conflict.
- A 'suicide' blast targets Pakistan police, killing at least five people and injuring 12.
- The IMF asks Pakistan to limit fuel subsidies.
- The Pakistan Army foils an infiltration bid at the Afghan border, killing eight militants.
- A crackdown on fancy number plates is underway.

Final Answer: The current top news in Pakistan today includes an air strike in Kabul, fuel price hikes, and a suicide blast targeting Pakistan police.


## **2. ReAct Agent**

In [ ]:
# Tools
search = DuckDuckGoSearchRun()

@tool
def get_weather_data(city: str) -> str:
    """Get current weather data for a given city."""
    url = f"https://api.weatherstack.com/current?access_key=3a62581f592ee671c2ab79172fc0bbe0&query={city}"
    response = requests.get(url)
    return response.json()


@tool
def search_tool(query: str) -> str:
    """Search the web for current information"""
    return search.run(query)

In [173]:
from langsmith import Client

client = Client()
react_prompt = client.pull_prompt("hwchase17/react")

In [174]:
llm = ChatGroq(
    model="llama-3.1-8b-instant"
)

agent = create_agent(
    model=llm,
    tools=[search_tool,get_weather_data],
    system_prompt=react_prompt.template,
)

In [175]:
response = agent.invoke({
    "messages": [
        {"role": "user", "content": "What is the Capital of Punjab,Pakistan and what is the weather condition it has now?"}]
})

print(response["messages"][-1].content)

The capital of Punjab, Pakistan is Lahore. 

The current weather condition in Lahore is smoke, rain with thunderstorm with a temperature of 22 degrees Celsius.
